# DSCI 511 Final Project:

## Research Question
**Which car performance and race strategy factors best predict a driver's final race finishing position?**

We collect telemetry and performance data from the [OpenF1 API](https://openf1.org/) across all 2023 Formula 1 race sessions, then combine it with historical race results from [StatsF1](https://www.statsf1.com) to build a unified dataset.

## OpenF1 Endpoints

| Endpoint | Description |
|---|---|
| `car_data` | High-frequency telemetry (speed, throttle, brake, RPM, gear, DRS) |
| `sessions` | Session metadata (race name, circuit, date) |
| `drivers` | Driver info per session |
| `laps` | Lap-by-lap timing data |
| `stints` | Tire stint info (compound, lap range) |
| `pit` | Pit stop events (lap number, duration) |
| `position` | Track position throughout session (aggregated to starting position) |
| `session_result` | Final classified results and grid positions |

In [1]:
import requests
import pandas as pd

response = requests.get(
  "https://api.openf1.org/v1/car_data",
  params={"driver_number": 1, "session_key": 9158}
)

df = pd.DataFrame(response.json())
print(f"Rows returned: {len(df)}")
df.head(10)

Rows returned: 18026


,date,session_key,throttle,speed,brake,rpm,meeting_key,driver_number,n_gear,drs
0,2023-09-15T09:15:02.731000+00:00,9158,0,0,0,0,1219,1,0,9
1,2023-09-15T09:15:02.971000+00:00,9158,0,0,0,0,1219,1,0,9
2,2023-09-15T09:15:03.211000+00:00,9158,0,0,0,0,1219,1,0,9
3,2023-09-15T09:15:03.491000+00:00,9158,0,0,0,0,1219,1,0,9
4,2023-09-15T09:15:03.771000+00:00,9158,0,0,0,0,1219,1,0,9
5,2023-09-15T09:15:04.092000+00:00,9158,0,0,0,0,1219,1,0,9
6,2023-09-15T09:15:04.492000+00:00,9158,0,0,0,0,1219,1,0,9
7,2023-09-15T09:15:04.732000+00:00,9158,0,0,0,0,1219,1,0,9
8,2023-09-15T09:15:05.012000+00:00,9158,0,0,0,0,1219,1,0,9
9,2023-09-15T09:15:05.292000+00:00,9158,0,0,0,0,1219,1,0,9


Initially, we started with setting the `SESSION_KEY = 9158` but after conducting some research, doing so would limit us to a single race, which is not enough data to answer our propsed research question "which factor predict finishin position". We would only have one data point per driver (20 rows total). We would need more variation across many races: different circuits, weather conditions, strategies, etc. The solution for now is to keep `SESSION_KEY` as a toggle while developing/testing. When `SESSION_KEY` is set we retrieve one session; when it is `None` we retrieve all race sessions for the year.

In [22]:
BASE_URL = "https://api.openf1.org/v1"
YEAR = 2023
SESSION_TYPE = "Race"
SESSION_KEY = None  # Set to None to collect all 2023 races

session_params = {"session_key": SESSION_KEY} if SESSION_KEY else {"year": YEAR, "session_type": SESSION_TYPE}

Before collecting data, we list all race sessions for the year to identify valid `session_key` values. This is useful for picking a specific session to test with, or to verify which sessions OpenF1 has coverage for.
The `sessions` endpoint returns metadata for each race session including circuit name, country, and date. This forms the backbone of our dataset, providing the race context that all other endpoints are joined against. 

In [23]:
response = requests.get(
    f"{BASE_URL}/sessions",
    params=session_params
)

df_sessions = pd.DataFrame(response.json())
print(f"Race sessions in {YEAR}: {len(df_sessions)}")
df_sessions[["session_key", "session_name", "date_start", "circuit_short_name", "country_name"]].head(10)

Race sessions in 2023: 29


,session_key,session_name,date_start,circuit_short_name,country_name
0,7953,Race,2023-03-05T15:00:00+00:00,Sakhir,Bahrain
1,7779,Race,2023-03-19T17:00:00+00:00,Jeddah,Saudi Arabia
2,7787,Race,2023-04-02T05:00:00+00:00,Melbourne,Australia
3,9069,Sprint,2023-04-29T13:30:00+00:00,Baku,Azerbaijan
4,9070,Race,2023-04-30T11:00:00+00:00,Baku,Azerbaijan
5,9078,Race,2023-05-07T19:30:00+00:00,Miami,United States
6,9086,Race,2023-05-21T13:00:00+00:00,Imola,Italy
7,9094,Race,2023-05-28T13:00:00+00:00,Monte Carlo,Monaco
8,9102,Race,2023-06-04T13:00:00+00:00,Catalunya,Spain
9,9110,Race,2023-06-18T18:00:00+00:00,Montreal,Canada


The `drivers` endpoint returns the list of drivers who participated in each session, along with their team and nationality. We deduplicate by `driver_number` and `session_key` since the raw API response can contain repeated entries for the same driver.

In [ ]:
all_drivers = []
for session_key in df_sessions["session_key"]:
    response = requests.get(
        f"{BASE_URL}/drivers", 
        params={"session_key": session_key}
    )
    data = response.json()
    if isinstance(data, list):
        all_drivers.extend(data)

df_drivers = pd.DataFrame(all_drivers).drop_duplicates(subset=["driver_number", "session_key"])
print(f"Driver records: {len(df_drivers)}")
df_drivers[["session_key", "driver_number", "full_name", "team_name", "country_code"]].head(10)

Driver records: 558


,session_key,driver_number,full_name,team_name,country_code
0,7953,1,Max VERSTAPPEN,Red Bull Racing,NED
1,7953,2,Logan SARGEANT,Williams,USA
2,7953,4,Lando NORRIS,McLaren,GBR
3,7953,10,Pierre GASLY,Alpine,FRA
4,7953,11,Sergio PEREZ,Red Bull Racing,MEX
5,7953,14,Fernando ALONSO,Aston Martin,ESP
6,7953,16,Charles LECLERC,Ferrari,MON
7,7953,18,Lance STROLL,Aston Martin,CAN
8,7953,20,Kevin MAGNUSSEN,Haas F1 Team,DEN
9,7953,21,Nyck DE VRIES,AlphaTauri,NED


The `laps` endpoint returns lap-by-lap timing data for every driver in a session, including lap duration, sector times, and tire compound. In the final dataset these are aggregated into per-driver summary statistics: average lap time, fastest lap, and lap time standard deviation (a measure of consistency).

In [ ]:
all_laps = []
for session_key in df_sessions["session_key"]:
    response = requests.get(
        f"{BASE_URL}/laps", 
        params={"session_key": session_key}
    )
    data = response.json()
    if isinstance(data, list):
        all_laps.extend(data)

df_laps = pd.DataFrame(all_laps)
print(f"Total lap records: {len(df_laps)}")
df_laps.head(5)

Total lap records: 26660


,meeting_key,session_key,driver_number,lap_number,date_start,duration_sector_1,duration_sector_2,duration_sector_3,i1_speed,i2_speed,is_pit_out_lap,lap_duration,segments_sector_1,segments_sector_2,segments_sector_3,st_speed
0,1141,7953,1,1,2023-03-05T15:03:38.500000+00:00,33.103,42.414,23.842,232.0,231.0,False,99.359,"[None, 2048, 2049, 2049, 2051, 2051, 2049, 204...","[2051, 2049, 2051, 2049, 2051, 2049, 2049, 204...","[2051, 2049, 2049, 2049, 2051, 2051]",252.0
1,1141,7953,11,1,2023-03-05T15:03:38.500000+00:00,34.120,43.216,24.069,230.0,237.0,False,101.405,"[None, 2048, 2049, 2049, 2049, 2049, 2049, 204...","[2049, 2049, 2049, 2049, 2049, 2049, 2049, 204...","[2049, 2049, 2049, 2049, 2049, 2049]",252.0
2,1141,7953,16,1,2023-03-05T15:03:38.500000+00:00,33.938,42.549,24.122,226.0,250.0,False,100.609,"[None, 2048, 2049, 2051, 2049, 2049, 2049, 204...","[2049, 2051, 2049, 2049, 2049, 2049, 2049, 205...","[2049, 2049, 2049, 2049, 2049, 2049]",255.0
3,1141,7953,31,1,2023-03-05T15:03:38.500000+00:00,36.463,45.213,24.698,233.0,238.0,False,106.374,"[2048, 2049, 2049, 2049, 2049, 2049, 2049, 204...","[2049, 2049, 2049, 2049, 2049, 2049, 2049, 204...","[2049, 2049, 2049, 2049, 2049, 2049, 2049]",254.0
4,1141,7953,27,1,2023-03-05T15:03:38.500000+00:00,37.038,45.924,24.741,220.0,239.0,False,107.703,"[2048, 2049, 2049, 2049, 2049, 2049, 2049, 204...","[2049, 2049, 2049, 2049, 2049, 2049, 2049, 204...","[2049, 2049, 2049, 2049, 2049, 2049, 2048]",249.0


A stint is a continuous run on one set of tires between pit stops. The `stints` endpoint tells us which tire compound each driver used, how many laps they ran on it, and how many stints they completed in total.

In [32]:
all_stints = []
for session_key in df_sessions["session_key"]:
    response = requests.get(
        f"{BASE_URL}/stints", 
        params={"session_key": session_key}
    )
    data = response.json()
    if isinstance(data, list):
        all_stints.extend(data)

df_stints = pd.DataFrame(all_stints)
print(f"Total stint records: {len(df_stints)}")
df_stints.head(5)

Total stint records: 1509


,meeting_key,session_key,stint_number,driver_number,lap_start,lap_end,compound,tyre_age_at_start
0,1141,7953,1,2,1.0,12.0,SOFT,0
1,1141,7953,1,22,1.0,10.0,SOFT,0
2,1141,7953,1,23,1.0,11.0,SOFT,0
3,1141,7953,1,63,1.0,13.0,SOFT,3
4,1141,7953,1,18,1.0,15.0,SOFT,3


The `pit` endpoint records each pit stop event, including the lap number and time spent in the pit lane. OpenF1's coverage of pit stop data is incomplete for some sessions — those return a 404 and are silently skipped by the `isinstance` check, so the loop is safe to run across all sessions.

In [37]:
all_pit = []
for session_key in df_sessions["session_key"]:
    response = requests.get(
        f"{BASE_URL}/pit", 
        params={"session_key": session_key}
    )
    data = response.json()
    if isinstance(data, list):
        all_pit.extend(data)

df_pit = pd.DataFrame(all_pit)
print(f"Total pit stop records: {len(df_pit)}")
df_pit.head(5)

Total pit stop records: 717


,date,session_key,pit_duration,meeting_key,driver_number,lap_number,lane_duration,stop_duration
0,2023-06-04T13:05:22.607000+00:00,9102,37.7,1211,4,1,37.7,None
1,2023-06-04T13:10:42.773000+00:00,9102,23.6,1211,77,5,23.6,None
2,2023-06-04T13:14:44.785000+00:00,9102,22.2,1211,27,8,22.2,None
3,2023-06-04T13:16:02.690000+00:00,9102,23.4,1211,24,9,23.4,None
4,2023-06-04T13:16:07.674000+00:00,9102,22.5,1211,21,9,22.5,None


The `position` endpoint tracks each driver's track position at sub-second frequency throughout the session, which is similar in volume to `car_data`. To make it usable in the final flat dataset, we aggregate immediately inside the loop by taking the first recorded position per driver per session, which approximates their starting grid position.

In [38]:
# Position data is sub-second frequency like car_data, so we aggregate immediately.
# We capture the starting position (first recorded position per driver per session).
all_position_agg = []
for session_key in df_sessions["session_key"]:
    response = requests.get(
        f"{BASE_URL}/position", 
        params={"session_key": session_key}
    )
    data = response.json()
    df_pos = pd.DataFrame(data) if isinstance(data, list) else pd.DataFrame()
    if not df_pos.empty:
        start_pos = (
            df_pos.sort_values("date")
            .groupby("driver_number")
            .first()[["position"]]
            .rename(columns={"position": "start_position"})
            .reset_index()
        )
        start_pos["session_key"] = session_key
        all_position_agg.append(start_pos)

df_position = pd.concat(all_position_agg, ignore_index=True) if all_position_agg else pd.DataFrame()
print(f"Position records (aggregated to start position per driver): {len(df_position)}")
df_position.head(10)

Position records (aggregated to start position per driver): 558


,driver_number,start_position,session_key
0,1,1,7953
1,2,16,7953
2,4,11,7953
3,10,20,7953
4,11,2,7953
5,14,5,7953
6,16,3,7953
7,18,8,7953
8,20,17,7953
9,21,19,7953


The `session_result` endpoint provides the official final classification for each session which are finishing position, championship points, and starting grid position. If this was a complete data science project, this would be the **target variable** for our research question: we aim to predict a driver's finishing `position` using the features collected from all other endpoints. It also serves as the base of the final merge, giving us one row per driver per race.

In [39]:
all_results = []
for session_key in df_sessions["session_key"]:
    response = requests.get(
        f"{BASE_URL}/session_result", 
        params={"session_key": session_key}
    )
    data = response.json()
    if isinstance(data, list):
        all_results.extend(data)

df_session_result = pd.DataFrame(all_results)
print(f"Total result records: {len(df_session_result)}")
df_session_result.head(10)

Total result records: 180


,position,driver_number,number_of_laps,points,dnf,dns,dsq,duration,gap_to_leader,meeting_key,session_key
0,1.0,1,57,25.0,False,False,False,5636.736,0,1141,7953
1,2.0,11,57,18.0,False,False,False,5648.723,11.987,1141,7953
2,3.0,14,57,15.0,False,False,False,5675.373,38.637,1141,7953
3,4.0,55,57,12.0,False,False,False,5684.788,48.052,1141,7953
4,5.0,44,57,10.0,False,False,False,5687.713,50.977,1141,7953
5,6.0,18,57,8.0,False,False,False,5691.238,54.502,1141,7953
6,7.0,63,57,6.0,False,False,False,5692.609,55.873,1141,7953
7,8.0,77,57,4.0,False,False,False,5709.383,72.647,1141,7953
8,9.0,10,57,2.0,False,False,False,5710.489,73.753,1141,7953
9,10.0,23,57,1.0,False,False,False,5726.510,89.774,1141,7953


The `car_data` endpoint returns high-frequency telemetry: speed, throttle, brake, RPM, gear, and DRS that were sampled multiple times per second. It is the heaviest endpoint with 18k+ rows per driver per session. Retrieving all its data during the full 2023 season without limiting any parameter would result in ~440 API calls, which might take over an hour. The proposed solution for now is instead of all 20 drivers, we only pick the top finishers (positions 1-5) to reduce the calls by 75%. Therefore, when we aggregate all the data in the end, drivers without `car_data` (positions 6-20) will get `Nan` for the `car_data` columns.

In [40]:
SAMPLE_DRIVERS = 5  # number of top finishers to fetch car_data for per session

all_car_data = []
for session_key in df_sessions["session_key"]:
    top_drivers = (
        df_session_result[df_session_result["session_key"] == session_key]
        .sort_values("position")
        .head(SAMPLE_DRIVERS)["driver_number"]
    )
    for driver_number in top_drivers:
        response = requests.get(
            f"{BASE_URL}/car_data", 
            params={"session_key": session_key, "driver_number": driver_number}
        )
        data = response.json()
        if isinstance(data, list):
            all_car_data.extend(data)

df_car_data = pd.DataFrame(all_car_data)
print(f"Total car data records: {len(df_car_data)}")
df_car_data.head(10)

Total car data records: 1459905


,date,session_key,n_gear,meeting_key,driver_number,drs,rpm,brake,speed,throttle
0,2023-03-05T14:01:02.639000+00:00,7953,0,1141,1,1,0,104,0,104
1,2023-03-05T14:01:02.999000+00:00,7953,0,1141,1,1,0,104,0,104
2,2023-03-05T14:01:03.279000+00:00,7953,0,1141,1,1,0,104,0,104
3,2023-03-05T14:01:03.439000+00:00,7953,0,1141,1,1,0,104,0,104
4,2023-03-05T14:01:03.679000+00:00,7953,0,1141,1,1,0,104,0,104
5,2023-03-05T14:01:03.879000+00:00,7953,0,1141,1,1,0,104,0,104
6,2023-03-05T14:01:04.959000+00:00,7953,0,1141,1,1,0,104,0,104
7,2023-03-05T14:01:05.199000+00:00,7953,0,1141,1,1,0,104,0,104
8,2023-03-05T14:01:05.399000+00:00,7953,0,1141,1,1,0,104,0,104
9,2023-03-05T14:01:05.639000+00:00,7953,0,1141,1,1,0,104,0,104


With all endpoints collected, we can build the final dataset. Each high-frequency source (`laps`, `pit`, `stints`, `car_data`) is first aggregated down to one summary row per driver per session, then joined together using `session_key` and `driver_number` as keys.

The base of every merge is `df_session_result` which has one row per driver per race. All other data is joined with left joins, so drivers with missing data for a given endpoint simply receive `NaN` rather than being dropped. The result is a flat dataset where each row represents one driver's complete performance profile for a single race, ready for exploratory analysis and predictive modelling.

In [41]:
# Aggregate laps per driver per session
df_laps_agg = (
    df_laps.groupby(["session_key", "driver_number"])
    .agg(
        total_laps=("lap_number", "max"),
        avg_lap_duration=("lap_duration", "mean"),
        fastest_lap=("lap_duration", "min"),
        lap_time_std=("lap_duration", "std"),
    )
    .reset_index()
)

# Aggregate pit stops per driver per session
df_pit_agg = (
    df_pit.groupby(["session_key", "driver_number"])
    .agg(
        pit_stop_count=("pit_duration", "count"),
        total_pit_time=("pit_duration", "sum"),
        avg_pit_duration=("pit_duration", "mean"),
    )
    .reset_index()
)

# Aggregate stints per driver per session
df_stints_agg = (
    df_stints.groupby(["session_key", "driver_number"])
    .agg(
        stint_count=("stint_number", "max"),
        compounds_used=("compound", lambda x: list(x.unique())),
    )
    .reset_index()
)

# Aggregate car_data per driver per session
df_car_data_agg = (
    df_car_data.groupby(["session_key", "driver_number"])
    .agg(
        avg_speed=("speed", "mean"),
        max_speed=("speed", "max"),
        avg_throttle=("throttle", "mean"),
        avg_brake=("brake", "mean"),
        avg_rpm=("rpm", "mean"),
        drs_usage=("drs", lambda x: (x >= 10).mean()),
    )
    .reset_index()
)

# Build combined dataset — one row per driver per race
df_combined = (
    df_session_result
    .merge(df_sessions[["session_key", "session_name", "circuit_short_name", "country_name", "date_start"]], on="session_key", how="left")
    .merge(df_drivers[["session_key", "driver_number", "full_name", "team_name", "country_code"]], on=["session_key", "driver_number"], how="left")
    .merge(df_laps_agg, on=["session_key", "driver_number"], how="left")
    .merge(df_pit_agg, on=["session_key", "driver_number"], how="left")
    .merge(df_stints_agg, on=["session_key", "driver_number"], how="left")
    .merge(df_position[["session_key", "driver_number", "start_position"]], on=["session_key", "driver_number"], how="left")
    .merge(df_car_data_agg, on=["session_key", "driver_number"], how="left")
)

print(f"Combined dataset shape: {df_combined.shape}")
print(f"Columns: {list(df_combined.columns)}")
df_combined.head(10)

Combined dataset shape: (180, 34)
Columns: ['position', 'driver_number', 'number_of_laps', 'points', 'dnf', 'dns', 'dsq', 'duration', 'gap_to_leader', 'meeting_key', 'session_key', 'session_name', 'circuit_short_name', 'country_name', 'date_start', 'full_name', 'team_name', 'country_code', 'total_laps', 'avg_lap_duration', 'fastest_lap', 'lap_time_std', 'pit_stop_count', 'total_pit_time', 'avg_pit_duration', 'stint_count', 'compounds_used', 'start_position', 'avg_speed', 'max_speed', 'avg_throttle', 'avg_brake', 'avg_rpm', 'drs_usage']


,position,driver_number,number_of_laps,points,dnf,dns,dsq,duration,gap_to_leader,meeting_key,...,avg_pit_duration,stint_count,compounds_used,start_position,avg_speed,max_speed,avg_throttle,avg_brake,avg_rpm,drs_usage
0,1.0,1,57,25.0,False,False,False,5636.736,0,1141,...,NaN,3.0,"[SOFT, HARD]",1,121.497629,309.0,47.076445,23.978594,6320.384362,0.000527
1,2.0,11,57,18.0,False,False,False,5648.723,11.987,1141,...,NaN,3.0,"[SOFT, HARD]",2,121.345654,331.0,47.361292,24.195314,6504.793179,0.007847
2,3.0,14,57,15.0,False,False,False,5675.373,38.637,1141,...,NaN,3.0,"[SOFT, HARD]",5,123.436379,329.0,52.276584,26.864051,6504.461472,0.017718
3,4.0,55,57,12.0,False,False,False,5684.788,48.052,1141,...,NaN,3.0,"[SOFT, HARD]",4,123.546596,329.0,53.014945,27.871011,6766.891834,0.005712
4,5.0,44,57,10.0,False,False,False,5687.713,50.977,1141,...,NaN,3.0,"[SOFT, HARD]",7,123.669402,334.0,48.500929,24.156717,6452.728268,0.016027
5,6.0,18,57,8.0,False,False,False,5691.238,54.502,1141,...,NaN,3.0,"[SOFT, HARD]",8,NaN,NaN,NaN,NaN,NaN,NaN
6,7.0,63,57,6.0,False,False,False,5692.609,55.873,1141,...,NaN,3.0,"[SOFT, HARD]",6,NaN,NaN,NaN,NaN,NaN,NaN
7,8.0,77,57,4.0,False,False,False,5709.383,72.647,1141,...,NaN,3.0,"[SOFT, HARD]",12,NaN,NaN,NaN,NaN,NaN,NaN
8,9.0,10,57,2.0,False,False,False,5710.489,73.753,1141,...,NaN,4.0,"[SOFT, HARD]",20,NaN,NaN,NaN,NaN,NaN,NaN
9,10.0,23,57,1.0,False,False,False,5726.510,89.774,1141,...,NaN,4.0,"[SOFT, HARD]",15,NaN,NaN,NaN,NaN,NaN,NaN
